# Evaluation against The Old Man and the Sea

In [1]:
import pandas as pd
import json
import pickle
import numpy as np

from sentiment_evaluation import SentimentEvaluator

In [2]:
with open('human_average_arc.pkl', 'rb') as f:
    human_average = pickle.load(f)
with open('machine_arcs.pkl', 'rb') as f:
    machine_arcs = pickle.load(f)

In [3]:
# linear scaler of roberta scores
def linear_scaling_relative (input_scores, new_range=(1, 10)):
    """
    relative linear scaler
    """
    values = np.array(input_scores, dtype=float)
    old_min = values.min()
    old_max = values.max()
    if old_min == old_max:
        raise ValueError("Cannot scale because all input values are the same.")
    scaled_values = np.interp(values, (old_min, old_max), new_range).tolist()
    return scaled_values

scaled_twt_roberta = linear_scaling_relative(machine_arcs['twt_roberta'])
scaled_twt_roberta_xlm = linear_scaling_relative(machine_arcs['twt_roberta_xlm'])

In [4]:
df_gemma = pd.read_csv("9 - Hemingway, Ernest - The Old Man and the Sea.csv")
teacher_model_scores = list(df_gemma['Sentiment_score'])

In [5]:
trivial_baseline_scores = [5 for _ in range(1923)]

In [6]:
evaluator = SentimentEvaluator(
    reference_scores=human_average,
    reference_label="Human Average",
    round_classes=False
)

In [7]:
evaluator.add_comparison(
    scores=teacher_model_scores,
    label="Gemma 4 31B (teacher model)"

)

evaluator.add_comparison(
        scores=scaled_twt_roberta,
        label=f"twt_roberta"
    )

evaluator.add_comparison(
        scores=scaled_twt_roberta_xlm,
        label=f"twt_roberta_xlm"
    )


for m in ['large', 'base', 'medium', 'mini', 'small', 'tiny']:
    df_interim = pd.read_json(f"bert_sentimnets/{m}.json")
    scores = list(df_interim['sentiment_label'])
    evaluator.add_comparison(
        scores=scores,
        label=f"BERT-{m}"
    )

evaluator.add_comparison(
    scores=trivial_baseline_scores,
    label='Trivial baseline'
)

In [8]:
results = evaluator.compare()

In [9]:
display(results)

,Reference,Model,N,N (non-neutral),MAE,Kendall's tau,Spearman's rho,Pearson's r,Precision (±1),Accuracy (±1),...,MAE (non-neutral),Spearman's rho (non-neutral),Pearson's r (non-neutral),"Precision (±1, non-neutral)","Accuracy (±1, non-neutral)",Sentiment precision (macro) (non-neutral),Precision Negative (non-neutral),Precision Neutral (non-neutral),Precision Positive (non-neutral),Sentiment accuracy (3-class) (non-neutral)
0,Human Average,Gemma 4 31B (teacher model),1923,305,0.821633,0.603957,0.690025,0.711352,0.460169,0.740510,...,1.116393,0.790749,0.884827,0.521541,0.600000,0.606260,0.547945,0.270833,1.000000,0.668852
1,Human Average,twt_roberta,1923,305,0.972425,0.497359,0.567890,0.629784,0.175715,0.695268,...,1.655724,0.744653,0.796565,0.306306,0.222951,0.623892,0.570248,0.321429,0.980000,0.636066
2,Human Average,twt_roberta_xlm,1923,305,1.261149,0.504668,0.592516,0.625922,0.125098,0.529901,...,1.612652,0.774664,0.837916,0.300575,0.308197,0.568619,0.519737,0.196429,0.989691,0.609836
3,Human Average,BERT-large,1923,305,0.768591,0.589415,0.675895,0.684946,0.613950,0.786791,...,0.886885,0.798533,0.876605,0.570929,0.711475,0.585234,0.538462,0.226415,0.990826,0.645902
4,Human Average,BERT-base,1923,305,0.732709,0.579568,0.665058,0.691877,0.723452,0.814873,...,0.726230,0.826135,0.898604,0.698311,0.803279,0.629279,0.569343,0.327586,0.990909,0.675410
5,Human Average,BERT-medium,1923,305,0.751950,0.488973,0.567642,0.623034,0.659088,0.807592,...,0.998361,0.784970,0.819763,0.669397,0.675410,0.604762,0.580645,0.259615,0.974026,0.570492
6,Human Average,BERT-mini,1923,305,0.765991,0.402016,0.472344,0.515533,0.731529,0.829433,...,0.968852,0.756338,0.784348,0.691896,0.708197,0.622908,0.588235,0.280488,1.000000,0.498361
7,Human Average,BERT-small,1923,305,0.717109,0.467591,0.541487,0.591258,0.708004,0.837233,...,0.981967,0.762674,0.793873,0.667814,0.698361,0.583615,0.558333,0.237288,0.955224,0.521311
8,Human Average,BERT-tiny,1923,305,0.596984,NaN,NaN,NaN,0.841394,0.841394,...,1.962295,NaN,NaN,0.000000,0.000000,0.083060,0.000000,0.249180,0.000000,0.249180
9,Human Average,Trivial baseline,1923,305,0.596984,NaN,NaN,NaN,0.841394,0.841394,...,1.962295,NaN,NaN,0.000000,0.000000,0.083060,0.000000,0.249180,0.000000,0.249180


In [10]:
evaluator.save_table("results/model_comparison.csv")

PosixPath('results/model_comparison.csv')